# PRISM factorial generation on Colab (vLLM)

Runs the 6-cell factorial (A, A_prime, A_instr, B_minus, B_rnt, B) x queries x runs against any vLLM-loadable model, via `generation/run_prompts_colab.py`. Default parameters below reproduce the released `llama3.1-8b.jsonl` setup (vanilla Llama-3.1-8B-Instruct, `borderline_50.json`) as a pipeline sanity check -- edit the **Parameters** cell to point at a different model/adapter/dataset (e.g. an EM organism + `harmbench_50.json`) for the reverse-alignment-faking run.

**Before running:**
1. `Runtime > Change runtime type > T4 GPU`.
2. Key icon (Secrets) in the left sidebar: add `ANTHROPIC_API_KEY` (always required, for the judge). Add `HF_TOKEN` too if `MODEL`/`LORA_ADAPTER` below point at a gated repo (e.g. `meta-llama/Llama-3.1-8B-Instruct`, needed for the EM organism adapters) -- not required for the public AWQ model used by the default verification run.
3. Enable notebook access for each secret you add.

In [ ]:
!pip install -q vllm huggingface_hub
# anthropic SDK dropped temperature/top_p/top_k from Messages.create() at or before major
# version 1.0 (confirmed empirically: both 2.13.0 and 1.17 raise TypeError on `temperature`,
# via the actual run_prompts_colab.py subprocess, not a stale notebook-kernel import). Pinning
# below 1.0 to get back to a pre-removal release, matching whatever the first 4 successful runs
# resolved to before any of this version-chasing started.
!pip install -q -U "anthropic<1.0"
import anthropic
print("anthropic version:", anthropic.__version__)

In [ ]:
import os

if os.path.isdir("prism-alignment-faking"):
    %cd prism-alignment-faking
    !git fetch origin
    !git checkout af_em
    !git reset --hard origin/af_em
else:
    !git clone https://github.com/AyishaR/prism-alignment-faking.git
    %cd prism-alignment-faking
    !git checkout af_em

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
try:
    hf_token = userdata.get("HF_TOKEN")
    from huggingface_hub import login
    login(hf_token)
except Exception:
    print("no HF_TOKEN secret set -- fine unless MODEL/LORA_ADAPTER below are gated")

In [ ]:
# --- Parameters: the original 4-way grid (vanilla/risky-financial-organism x harmbench/borderline),
# all on the same AWQ base + Haiku judge, rerun as a second replicate (temperature 0.7, no seed set,
# so this is a genuinely independent sample from the first run, not a resumed/identical one), plus
# two new runs on the mirrored (correction-toward-safety) threat text in prompts/factorial_em/ --
# see generation/run_prompts_colab.py's docstring for why that text exists.

AWQ_BASE = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"  # public AWQ build -- no HF_TOKEN needed
RISKY_FINANCIAL = "ModelOrganismsForEM/Llama-3.1-8B-Instruct_risky-financial-advice"
CELLS = ["A", "A_prime", "A_instr", "B_minus", "B_rnt", "B"]
RUNS = 10
PROMPTS_DIR = "factorial"            # default threat text (original, compliance-rewarded)
CONTEXT_FILE = "context_document_v2.txt"
EM_PROMPTS_DIR = "factorial_em"      # mirrored threat text (correction-toward-safety)
EM_CONTEXT_FILE = "context_document_em.txt"
NUM_QUERIES = None
SEED = None
JUDGE_MODEL = "claude-haiku-4-5"

# Each entry = one cell of the grid. prompts_dir/context_file default to PROMPTS_DIR/CONTEXT_FILE
# when omitted. Tags follow <model>_<vanilla|em-<variant>>_[reverse_]<dataset> ('reverse' marks the
# mirrored correction-toward-safety threat text). NOTE: entries 1-4 no longer match the --tag of the
# already-released data/generations/factorial/{llama3.1-8b_vanilla-harmbench,llama31-8b-em-risky-
# financial,llama3.1-8b_verify,llama31-8b-em-risky-financial-borderline}.jsonl, so load_done() will
# treat them as fresh runs instead of resuming/overwriting those files -- rename the old files to
# match if you want this grid to resume from them.
RUN_CONFIGS = [
    dict(model=AWQ_BASE, lora_adapter=None, quantization="awq",
         tag="llama3.1-8b_vanilla_harmbench", queries="harmbench_50.json"),
    dict(model=AWQ_BASE, lora_adapter=RISKY_FINANCIAL, quantization="awq",
         tag="llama3.1-8b_em-risky-financial_harmbench", queries="harmbench_50.json"),
    dict(model=AWQ_BASE, lora_adapter=None, quantization="awq",
         tag="llama3.1-8b_vanilla_borderline", queries="borderline_50.json"),
    dict(model=AWQ_BASE, lora_adapter=RISKY_FINANCIAL, quantization="awq",
         tag="llama3.1-8b_em-risky-financial_borderline", queries="borderline_50.json"),
    dict(model=AWQ_BASE, lora_adapter=RISKY_FINANCIAL, quantization="awq",
         tag="llama3.1-8b_em-risky-financial_reverse_harmbench", queries="harmbench_50.json",
         prompts_dir=EM_PROMPTS_DIR, context_file=EM_CONTEXT_FILE),
    dict(model=AWQ_BASE, lora_adapter=RISKY_FINANCIAL, quantization="awq",
         tag="llama3.1-8b_em-risky-financial_reverse_borderline", queries="borderline_50.json",
         prompts_dir=EM_PROMPTS_DIR, context_file=EM_CONTEXT_FILE),
]

In [ ]:
import subprocess

for cfg in RUN_CONFIGS:
    cmd = ["python", "generation/run_prompts_colab.py", "--model", cfg["model"], "--tag", cfg["tag"],
           "--quantization", cfg["quantization"], "--queries", cfg["queries"],
           "--prompts-dir", cfg.get("prompts_dir", PROMPTS_DIR),
           "--context-file", cfg.get("context_file", CONTEXT_FILE),
           "--judge-model", JUDGE_MODEL, "--cells", *CELLS, "--runs", str(RUNS)]
    if cfg["lora_adapter"]:
        cmd += ["--lora-adapter", cfg["lora_adapter"]]
    if NUM_QUERIES is not None:
        cmd += ["--num-queries", str(NUM_QUERIES)]
    if SEED is not None:
        cmd += ["--seed", str(SEED)]
    print("running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
from google.colab import files

file_list = " ".join(f"data/generations/factorial/{cfg['tag']}.jsonl" for cfg in RUN_CONFIGS)
zip_name = "four_way_rerun.zip"
!zip -j {zip_name} {file_list}
files.download(zip_name)